# EEEM068 — Knee Abnormality ViT (MRNet)
**Workflow:** Edit code in VS Code → push to GitHub → run this notebook on Colab GPU → results saved to Drive

## Phase 1 — Setup & Environment

In [ ]:
GITHUB_REPO      = "https://github.com/sachithgowda-cloud/knee-abnormality-vit.git"
DRIVE_BASE       = "/content/drive/MyDrive/knee-abnormality-vit"
OUTPUT_DIR       = f"{DRIVE_BASE}/outputs"
SIT_WEIGHTS_PATH = f"{DRIVE_BASE}/weights/sit-s.pth"
REPO_DIR         = "/content/knee-abnormality-vit"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, os
from pathlib import Path

EXTRACT_DIR = "/content/mrnet_data"

# Check if already extracted from a previous run
already_extracted = list(Path(EXTRACT_DIR).rglob("train-acl.csv")) if Path(EXTRACT_DIR).exists() else []

if already_extracted:
    MRNET_DATA_DIR = str(already_extracted[0].parent)
    print(f"Already extracted: {MRNET_DATA_DIR}")
else:
    # Find archive.zip anywhere in Drive
    print("Searching for archive.zip in Google Drive...")
    result = subprocess.run(
        ["find", "/content/drive/MyDrive", "-name", "archive.zip", "-type", "f"],
        capture_output=True, text=True, timeout=60
    )
    zips = [l.strip() for l in result.stdout.strip().splitlines() if l.strip()]

    if not zips:
        raise FileNotFoundError("Could not find archive.zip in your Drive.")

    zip_path = zips[0]
    print(f"Found: {zip_path}")
    print("Extracting... (takes ~2 min for 5GB)")

    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{zip_path}" -d {EXTRACT_DIR}

    csvs = list(Path(EXTRACT_DIR).rglob("train-acl.csv"))
    if not csvs:
        raise FileNotFoundError(f"Extracted zip but could not find train-acl.csv inside {EXTRACT_DIR}")

    MRNET_DATA_DIR = str(csvs[0].parent)
    print(f"Dataset ready at: {MRNET_DATA_DIR}")

In [ ]:
import os

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {GITHUB_REPO} {REPO_DIR}

os.chdir(REPO_DIR)
print("Repo dir:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### SiT-S Pretrained Weights (Sara-Ahmed/SiT)

In [ ]:
import os
from pathlib import Path

# Clone SiT repo to get the vision_transformer.py architecture
SIT_REPO = "/content/SiT"
if not os.path.exists(SIT_REPO):
    !git clone https://github.com/Sara-Ahmed/SiT.git {SIT_REPO}
    print("Cloned SiT repo")
else:
    print("SiT repo already present")

# Download SiT pretrained weights from official Drive folder
os.makedirs(os.path.dirname(SIT_WEIGHTS_PATH), exist_ok=True)

if Path(SIT_WEIGHTS_PATH).exists() and Path(SIT_WEIGHTS_PATH).stat().st_size > 1_000_000:
    print("SiT weights already present:", SIT_WEIGHTS_PATH)
else:
    print("Downloading SiT-S weights from Sara-Ahmed/SiT Drive folder...")
    # Official folder: https://drive.google.com/drive/folders/11lGoNZKcMr6A959Yun_MrSlT3j6h-4YI
    WEIGHTS_TMP = "/content/sit_weights"
    os.makedirs(WEIGHTS_TMP, exist_ok=True)
    !gdown --folder "https://drive.google.com/drive/folders/11lGoNZKcMr6A959Yun_MrSlT3j6h-4YI" -O {WEIGHTS_TMP} --quiet

    # Find the SiT-S checkpoint (small model)
    downloaded = list(Path(WEIGHTS_TMP).rglob("*.pth")) + list(Path(WEIGHTS_TMP).rglob("*.pt"))
    print("Downloaded files:", [f.name for f in downloaded])

    sit_s_file = next((f for f in downloaded if "SiT-S" in f.name or "sit-s" in f.name.lower() or "small" in f.name.lower()), None)
    if sit_s_file is None and downloaded:
        sit_s_file = downloaded[0]   # fallback: take first available

    if sit_s_file:
        import shutil
        shutil.copy(sit_s_file, SIT_WEIGHTS_PATH)
        print(f"Saved SiT-S weights to: {SIT_WEIGHTS_PATH}")
    else:
        print("WARNING: Could not find SiT weights — falling back to timm pretrained")
        SIT_WEIGHTS_PATH = None

print("SIT_WEIGHTS_PATH =", SIT_WEIGHTS_PATH)

## Phase 2 — Data Pipeline

In [ ]:
from pathlib import Path

data_root = Path(MRNET_DATA_DIR)

print("Dataset structure:")
for split in ("train", "valid"):
    for plane in ("axial", "coronal", "sagittal"):
        folder = data_root / split / plane
        n = len(list(folder.glob("*.npy"))) if folder.exists() else 0
        print(f"  {split}/{plane}: {n} volumes")

In [ ]:
import pandas as pd

for task in ("acl", "meniscus", "abnormal"):
    df = pd.read_csv(data_root / f"train-{task}.csv", header=None, names=["case", task])
    print(f"train-{task}: {df[task].sum()} positive / {len(df)} total")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, REPO_DIR)

sample_vol = np.load(data_root / "train" / "sagittal" / "0000.npy")
print(f"Volume shape: {sample_vol.shape}  (slices x H x W)")
print(f"Pixel range : {sample_vol.min():.1f} - {sample_vol.max():.1f}")

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
n = sample_vol.shape[0]
for i, ax in enumerate(axes):
    ax.imshow(sample_vol[n * i // 5], cmap="gray")
    ax.set_title(f"Slice {n * i // 5}")
    ax.axis("off")
plt.suptitle("Sample sagittal MRI volume")
plt.tight_layout()
plt.show()

In [ ]:
import yaml
from src.dataset import get_dataloaders, MRNetDataset

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

loaders, classes = get_dataloaders(MRNET_DATA_DIR, cfg)

print(f"Classes : {classes}")
for split, loader in loaders.items():
    print(f"{split:5s}: {len(loader.dataset):4d} samples, {len(loader):3d} batches")

In [ ]:
import torchvision

images, labels = next(iter(loaders["train"]))
print(f"Batch shape: {images.shape}")

mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
imgs_display = (images[:8] * std + mean).clamp(0, 1)

grid = torchvision.utils.make_grid(imgs_display, nrow=4)
plt.figure(figsize=(12, 6))
plt.imshow(grid.permute(1, 2, 0))
plt.title("Augmented training batch - labels: " + str(labels[:8].tolist()))
plt.axis("off")
plt.show()

## Phase 3 — Model (run after Phase 2 is verified)

In [ ]:
from src.model import build_model, get_optimizer
from pathlib import Path

sit_path = SIT_WEIGHTS_PATH if Path(SIT_WEIGHTS_PATH).exists() else None

model = build_model(cfg, sit_weights_path=sit_path).to(device)
optimizer = get_optimizer(model, cfg)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total/1e6:.1f}M total, {trainable/1e6:.1f}M trainable")

## Phase 4 — Training

In [ ]:
from src.trainer import train
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

history = train(model, loaders, cfg, output_dir, device, optimizer)
print("Best checkpoint:", output_dir / "best_model.pth")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/tensorboard

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="train")
ax1.plot(history["val_loss"],   label="val")
ax1.set_title("Loss"); ax1.legend()
ax2.plot(history["train_acc"],  label="train")
ax2.plot(history["val_acc"],    label="val")
ax2.set_title("Accuracy"); ax2.legend()
plt.tight_layout()
plt.savefig(output_dir / "training_curves.png", dpi=150)
plt.show()

## Script-Based Colab Tests
Run these cells after the setup cells above if you want to use the newer CLI workflows added to the repo.


In [ ]:
%cd /content/knee-abnormality-vit
!git pull origin main
!python3 scripts/train.py --colab --sit-weights {SIT_WEIGHTS_PATH}


### Evaluate The Best Checkpoint
Replace `RUN_DIR` with the training run folder you want to evaluate.


In [ ]:
RUN_DIR = f"{OUTPUT_DIR}/run_YYYYMMDD_HHMMSS"
!python3 scripts/evaluate.py --colab --checkpoint {RUN_DIR}/best_model.pth


### Attention Maps
Generate attention overlays for correctly classified validation slices.


In [ ]:
!python3 scripts/visualize_attention.py --colab --checkpoint {RUN_DIR}/best_model.pth --selection correct --num-samples 12 --output-dir {RUN_DIR}/attention_correct
